# This is a simple example script for 2D x/y stitching using muvis-align and the multiview-stitcher package

In [ ]:
from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import vis_utils

from src.muvis_align.MVSRegistration import MVSRegistration
from src.muvis_align.image.util import get_sim_physical_size, show_image

## Initialise muvis-align, initialise sims, and pre-process

In [ ]:
reg = MVSRegistration(operation='register', input_pattern='../data/S000/*.zarr', output_pattern='../../output/', ui='mpl', debug=True)
sims = reg.init_sims()
reg_sims, reg_indices = reg.preprocess(sims)

for label, sim in zip(reg.file_labels, sims):
    print(label, si_utils.get_origin_from_sim(sim), get_sim_physical_size(sim))

## Initialise registration parameters

In [ ]:
register_params = {
	'pairing': 'orthogonal',
	'transform_type': 'affine',
	'registration':
	{
		'name': 'sift',
		'gaussian_sigma': 2,
		'normalisation': True,
		'max_keypoints': 5000,
		'inlier_threshold_factor': 0.05,
		'max_trials': 1000,
		'ransac_iterations': 3
	},
	'n_parallel_pairwise_regs': 1,
}

## Perform registration (using multiview-stitcher)

In [ ]:
%matplotlib inline
results = reg.register(sims, reg_sims, reg_indices, register_params)
registered_sims = results['sims']

## Show registration mapping

In [ ]:
mappings = results['mappings']
for label, mapping in zip(reg.file_labels, mappings.values()):
    print(f'{label}:\n', mapping.sel(t=0).data)

## Visualise registered sims

In [ ]:
%matplotlib inline
fig, ax = vis_utils.plot_positions(registered_sims, transform_key=reg.reg_transform_key, use_positional_colors=False, view_labels=reg.file_labels)

## Perform fusion (using multiview-stitcher)

In [ ]:
%matplotlib inline
fused, _ = reg.fuse(registered_sims)

fused

## Output fused result

In [ ]:
%matplotlib inline
show_image(fused[0, 0])

In [ ]:
reg.save(reg.output + 'stitching2d', fused)